In [11]:
import os
import pandas as pd
import numpy as np
import glob
from collections import defaultdict



In [12]:
current_dir = os.getcwd()
dataset = "result_blogcatalog"
dataset_folders = [
    name for name in os.listdir(current_dir)
    if os.path.isdir(os.path.join(current_dir, name)) and dataset in name.lower()
]

dataset_folders

['result_blogcatalog_gcn',
 'result_blogcatalog_gin',
 'result_blogcatalog_svdgcn',
 'result_blogcatalog_rgcn',
 'result_blogcatalog_jacgcn',
 'result_blogcatalog_graphsage']

In [20]:
result_csv_paths = {}
dataset_name = dataset.split('_')[-1]
result_csv_paths[dataset_name] = {}
for folder in dataset_folders:
    defance_model = folder.split('_')[-1]
    csv_paths = glob.glob(f"{current_dir}/{folder}/*.csv")
    
    groups = defaultdict(list)
    for path in csv_paths:
        fname = os.path.basename(path)
        name_no_ext = os.path.splitext(fname)[0]
        attack_name = name_no_ext.split("_")[0].lower()

        groups[attack_name].append(os.path.abspath(path))
    
    result_csv_paths[dataset_name][defance_model] = dict(groups)

print(result_csv_paths)

{'blogcatalog': {'gcn': {'sgattack': ['/home/munem/codes/graph-attack/result_blogcatalog_gcn/SGAttack_blogcatalog_gcn_3.csv', '/home/munem/codes/graph-attack/result_blogcatalog_gcn/SGAttack_blogcatalog_gcn_1.csv', '/home/munem/codes/graph-attack/result_blogcatalog_gcn/SGAttack_blogcatalog_gcn_5.csv', '/home/munem/codes/graph-attack/result_blogcatalog_gcn/SGAttack_blogcatalog_gcn_4.csv', '/home/munem/codes/graph-attack/result_blogcatalog_gcn/SGAttack_blogcatalog_gcn_2.csv'], 'proposed': ['/home/munem/codes/graph-attack/result_blogcatalog_gcn/proposed_model_blogcatalog_gcn_5.csv', '/home/munem/codes/graph-attack/result_blogcatalog_gcn/proposed_model_blogcatalog_gcn_1.csv', '/home/munem/codes/graph-attack/result_blogcatalog_gcn/proposed_model_blogcatalog_gcn_3.csv', '/home/munem/codes/graph-attack/result_blogcatalog_gcn/proposed_model_blogcatalog_gcn_4.csv', '/home/munem/codes/graph-attack/result_blogcatalog_gcn/proposed_model_blogcatalog_gcn_2.csv'], 'random': ['/home/munem/codes/graph-a

In [21]:
results_summary = {}
attack_name_map = {
    "proposed": "GAEttack",
    "sgattack": "SGAttack",
    "fga": "FGA",
    "nettack": "Nettack",
    "random": "Random"
}

for dataset, dataset_info in result_csv_paths.items():
    results_summary[dataset] = {}

    for defance_model, defance_model_info in dataset_info.items():
        results_summary[dataset][defance_model] = {}

        for attack_name, attack_csv_paths in defance_model_info.items():
            # Normalize and rename attack_name if it exists in the map
            attack_name_clean = attack_name_map.get(attack_name.lower(), attack_name)

            # Collect all dataframes for this attack
            dfs = []
            for path in attack_csv_paths:
                df = pd.read_csv(path)
                df = df[["budget_number", "miss-classification_modified"]]
                dfs.append(df)

            # Combine all CSVs for this attack
            combined = pd.concat(dfs, ignore_index=True)

            # Group by budget and compute mean & std
            grouped = (
                combined.groupby("budget_number")["miss-classification_modified"]
                .agg(["mean", "std"])
                .reset_index()
            )

            # Format as "mean ± std"
            formatted = [
                f"{row['mean']:.3f} ± {row['std']:.3f}"
                for _, row in grouped.iterrows()
            ]

            # Store in nested dictionary
            results_summary[dataset][defance_model][attack_name_clean] = formatted



In [22]:
results_summary

{'blogcatalog': {'gcn': {'SGAttack': ['0.195 ± 0.048',
    '0.270 ± 0.011',
    '0.330 ± 0.045',
    '0.385 ± 0.060',
    '0.410 ± 0.049',
    '0.440 ± 0.049',
    '0.460 ± 0.045'],
   'GAEttack': ['0.180 ± 0.033',
    '0.275 ± 0.025',
    '0.345 ± 0.065',
    '0.395 ± 0.099',
    '0.405 ± 0.108',
    '0.425 ± 0.092',
    '0.450 ± 0.094'],
   'Random': ['0.140 ± 0.055',
    '0.125 ± 0.059',
    '0.165 ± 0.045',
    '0.160 ± 0.055',
    '0.180 ± 0.037',
    '0.205 ± 0.037',
    '0.200 ± 0.056'],
   'Nettack': ['0.210 ± 0.034',
    '0.305 ± 0.027',
    '0.340 ± 0.022',
    '0.385 ± 0.080',
    '0.455 ± 0.074',
    '0.465 ± 0.060',
    '0.510 ± 0.068'],
   'FGA': ['0.155 ± 0.065',
    '0.215 ± 0.084',
    '0.240 ± 0.055',
    '0.260 ± 0.080',
    '0.330 ± 0.065',
    '0.340 ± 0.060',
    '0.360 ± 0.055']},
  'gin': {'FGA': ['0.770 ± 0.082',
    '0.775 ± 0.105',
    '0.745 ± 0.065',
    '0.745 ± 0.054',
    '0.790 ± 0.055',
    '0.755 ± 0.069',
    '0.760 ± 0.022'],
   'GAEttack': ['0.785 

In [24]:
import re
import json

# Helper: extract mean value from "mean ± std" string
def extract_mean(value_str):
    match = re.match(r"([\d.]+)", value_str)
    return float(match.group(1)) if match else float('nan')

highest_summary = {}

for dataset, defense_info in results_summary.items():
    highest_summary[dataset] = {}

    for defense_model, attacks_info in defense_info.items():
        highest_summary[dataset][defense_model] = {}

        # Determine number of budgets (max list length among attacks)
        n_budgets = max(len(v) for v in attacks_info.values())

        for i in range(n_budgets):
            # Collect all (attack_name, mean_value) for this budget index
            values = []
            for attack_name, stats_list in attacks_info.items():
                if i < len(stats_list):
                    mean_val = extract_mean(stats_list[i])
                    values.append((attack_name, mean_val))

            # Sort descending by mean value
            values.sort(key=lambda x: x[1], reverse=True)

            # Get top two (if available)
            top1 = values[0] if len(values) > 0 else (None, None)
            top2 = values[1] if len(values) > 1 else (None, None)

            highest_summary[dataset][defense_model][f"budget_{i+1}"] = {
                "highest": {
                    "attack_name": top1[0],
                    "value": round(top1[1], 3) if top1[1] is not None else None,
                    "index": i
                },
                "second_highest": {
                    "attack_name": top2[0],
                    "value": round(top2[1], 3) if top2[1] is not None else None,
                    "index": i
                }
            }

# Display results
print(json.dumps(highest_summary, indent=2))



{
  "blogcatalog": {
    "gcn": {
      "budget_1": {
        "highest": {
          "attack_name": "Nettack",
          "value": 0.21,
          "index": 0
        },
        "second_highest": {
          "attack_name": "SGAttack",
          "value": 0.195,
          "index": 0
        }
      },
      "budget_2": {
        "highest": {
          "attack_name": "Nettack",
          "value": 0.305,
          "index": 1
        },
        "second_highest": {
          "attack_name": "GAEttack",
          "value": 0.275,
          "index": 1
        }
      },
      "budget_3": {
        "highest": {
          "attack_name": "GAEttack",
          "value": 0.345,
          "index": 2
        },
        "second_highest": {
          "attack_name": "Nettack",
          "value": 0.34,
          "index": 2
        }
      },
      "budget_4": {
        "highest": {
          "attack_name": "GAEttack",
          "value": 0.395,
          "index": 3
        },
        "second_highest": {
      

In [26]:
import re
import copy
import json

# Helper to extract mean from "mean ± std"
def extract_mean(value_str):
    match = re.match(r"([\d.]+)", value_str)
    return float(match.group(1)) if match else float('-inf')

# Deep copy the original results_summary to preserve structure
annotated_summary = copy.deepcopy(results_summary)

for dataset, defense_info in results_summary.items():
    for defense_model, attacks_info in defense_info.items():
        # Determine number of budgets
        n_budgets = max(len(v) for v in attacks_info.values())

        for i in range(n_budgets):
            # Collect (attack_name, mean_value) for this budget index
            values = []
            for attack_name, stats_list in attacks_info.items():
                if i < len(stats_list):
                    mean_val = extract_mean(stats_list[i])
                    values.append((attack_name, mean_val))

            # Sort descending by mean value
            values.sort(key=lambda x: x[1], reverse=True)
            top1_name = values[0][0] if len(values) > 0 else None
            top2_name = values[1][0] if len(values) > 1 else None

            # Annotate in the copied dict
            for attack_name, stats_list in attacks_info.items():
                if i < len(stats_list):
                    suffix = ""
                    if attack_name == top1_name:
                        suffix = " (highest)"
                    elif attack_name == top2_name:
                        suffix = " (second highest)"
                    # Update the copied dict
                    annotated_summary[dataset][defense_model][attack_name][i] += suffix

# Print annotated results
print(json.dumps(annotated_summary, indent=2, ensure_ascii=False))


{
  "blogcatalog": {
    "gcn": {
      "SGAttack": [
        "0.195 ± 0.048 (second highest)",
        "0.270 ± 0.011",
        "0.330 ± 0.045",
        "0.385 ± 0.060 (second highest)",
        "0.410 ± 0.049 (second highest)",
        "0.440 ± 0.049 (second highest)",
        "0.460 ± 0.045 (second highest)"
      ],
      "GAEttack": [
        "0.180 ± 0.033",
        "0.275 ± 0.025 (second highest)",
        "0.345 ± 0.065 (highest)",
        "0.395 ± 0.099 (highest)",
        "0.405 ± 0.108",
        "0.425 ± 0.092",
        "0.450 ± 0.094"
      ],
      "Random": [
        "0.140 ± 0.055",
        "0.125 ± 0.059",
        "0.165 ± 0.045",
        "0.160 ± 0.055",
        "0.180 ± 0.037",
        "0.205 ± 0.037",
        "0.200 ± 0.056"
      ],
      "Nettack": [
        "0.210 ± 0.034 (highest)",
        "0.305 ± 0.027 (highest)",
        "0.340 ± 0.022 (second highest)",
        "0.385 ± 0.080",
        "0.455 ± 0.074 (highest)",
        "0.465 ± 0.060 (highest)",
        "0.

## Time

In [12]:
import json
import glob
import numpy as np
from pathlib import Path

# Define the file path pattern
file_path = "running_time_*_dataset_*_defense_*.json"  # Adjust this path as needed

# Attack name mapping
attack_name_mapping = {
    "Proposed_model": "GAEttack",
    "Random_attack": "Random",
    "FGA": "FGA",
    "Nettack": "Nettack",
    "SGAttack_attack": "SGAttack"
}

def parse_time_to_seconds(time_str):
    """Convert time string to total seconds"""
    parts = time_str.split(", ")
    minutes = int(parts[0].split()[0])
    seconds = float(parts[1].split()[0])
    return minutes * 60 + seconds

def process_files(file_pattern):
    # Dictionary to store all data
    # Structure: {dataset: {attack_name: {budget: [list of times]}}}
    data = {}
    
    # Glob all matching files
    files = glob.glob(file_pattern)
    
    if not files:
        print(f"No files found matching pattern: {file_pattern}")
        return None
    
    print(f"Found {len(files)} files")
    
    # Process each file
    for file in files:
        # Extract dataset name from filename
        filename = Path(file).stem
        parts = filename.split("_")
        
        # Find the index of "dataset" to get the dataset name
        dataset_idx = parts.index("dataset") + 1
        dataset_name = parts[dataset_idx]
        
        print(f"Processing {file} - Dataset: {dataset_name}")
        
        # Read JSON file
        with open(file, 'r') as f:
            json_data = json.load(f)
        
        # Initialize dataset in data dictionary if not exists
        if dataset_name not in data:
            data[dataset_name] = {}
        
        # Process each attack type
        for original_attack_name, attack_data in json_data.items():
            # Map to new attack name
            new_attack_name = attack_name_mapping.get(original_attack_name, original_attack_name)
            
            # Initialize attack name in dataset if not exists
            if new_attack_name not in data[dataset_name]:
                data[dataset_name][new_attack_name] = {}
            
            # Process each budget entry
            for entry in attack_data:
                budget = entry["budget"]
                running_time = entry["running_time"]
                
                # Convert time to seconds
                time_seconds = parse_time_to_seconds(running_time)
                
                # Initialize budget list if not exists
                budget_key = f"budget_{budget}"
                if budget_key not in data[dataset_name][new_attack_name]:
                    data[dataset_name][new_attack_name][budget_key] = []
                
                # Append time to the list
                data[dataset_name][new_attack_name][budget_key].append(time_seconds)
    
    return data

def calculate_statistics(data):
    """Calculate average ± std for each budget"""
    result = {}
    
    for dataset, attacks in data.items():
        result[dataset] = {}
        
        for attack_name, budgets in attacks.items():
            result[dataset][attack_name] = {}
            
            for budget_key, times in budgets.items():
                # Calculate mean and std
                mean_time = np.mean(times)
                std_time = np.std(times, ddof=1) if len(times) > 1 else 0.0
                
                # Format as "avg ± std" with 2 decimal places
                result[dataset][attack_name][budget_key] = f"{mean_time:.2f} ± {std_time:.2f}"
    
    return result

def main():
    # Process files (adjust the path pattern as needed)
    file_pattern = "./running_time_result/running_time_*_dataset_*_defense_*.json"
    
    print("Starting to process files...")
    data = process_files(file_pattern)
    
    if data is None:
        print("No data processed. Check your file pattern.")
        return
    
    # Calculate statistics
    print("\nCalculating statistics...")
    result = calculate_statistics(data)
    
    # Save to output JSON file
    output_file = "running_time_statistics.json"
    with open(output_file, 'w') as f:
        json.dump(result, f, indent=4, ensure_ascii=False)
    
    print(f"\nResults saved to {output_file}")
    
    # Print summary
    print("\nSummary:")
    for dataset in result:
        print(f"  Dataset: {dataset}")
        for attack in result[dataset]:
            print(f"    Attack: {attack}")
            budgets = list(result[dataset][attack].keys())
            print(f"      Budgets: {len(budgets)} ({min(budgets)} to {max(budgets)})")

if __name__ == "__main__":
    main()

Starting to process files...
Found 5 files
Processing ./running_time_result/running_time_2_dataset_blogcatalog_defense_gcn.json - Dataset: blogcatalog
Processing ./running_time_result/running_time_1_dataset_blogcatalog_defense_gcn.json - Dataset: blogcatalog
Processing ./running_time_result/running_time_3_dataset_blogcatalog_defense_gcn.json - Dataset: blogcatalog
Processing ./running_time_result/running_time_4_dataset_blogcatalog_defense_gcn.json - Dataset: blogcatalog
Processing ./running_time_result/running_time_5_dataset_blogcatalog_defense_gcn.json - Dataset: blogcatalog

Calculating statistics...

Results saved to running_time_statistics.json

Summary:
  Dataset: blogcatalog
    Attack: GAEttack
      Budgets: 7 (budget_1 to budget_7)
    Attack: Random
      Budgets: 7 (budget_1 to budget_7)
    Attack: FGA
      Budgets: 7 (budget_1 to budget_7)
    Attack: Nettack
      Budgets: 7 (budget_1 to budget_7)
    Attack: SGAttack
      Budgets: 7 (budget_1 to budget_7)


In [28]:
import json
import matplotlib.pyplot as plt
import numpy as np

def create_attack_plot(data, output_filename='plot.pdf', show_legend=True):
    """
    Create high-quality misclassification plot from JSON data.
    
    Parameters:
    -----------
    data : dict
        Dictionary containing plot data with keys:
        - 'title': str (optional, e.g., 'GIN on Polblogs')
        - 'budgets': list of int/float (x-axis values)
        - 'attacks': dict with attack names as keys and misclassification values as values
        - 'ylabel': str (optional, default 'Misclassification (%)')
        - 'xlabel': str (optional, default 'Perturbation Budget')
        - 'show_legend': bool (optional, overrides show_legend parameter)
    output_filename : str
        Output PDF filename
    show_legend : bool
        Whether to display the legend (default: True)
    """
    
    # Set high-quality plot parameters
    plt.rcParams['figure.dpi'] = 300
    plt.rcParams['savefig.dpi'] = 300
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif']
    # plt.rcParams['font.size'] = 13
    plt.rcParams['font.weight'] = 'bold'
    plt.rcParams['axes.linewidth'] = 1.5
    plt.rcParams['axes.labelweight'] = 'bold'
    plt.rcParams['lines.linewidth'] = 2.5
    plt.rcParams['lines.markersize'] = 9
    plt.rcParams['xtick.major.width'] = 1.5
    plt.rcParams['ytick.major.width'] = 1.5
    # plt.rcParams['xtick.labelsize'] = 12
    # plt.rcParams['ytick.labelsize'] = 12
    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['ytick.direction'] = 'in'
    # plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['legend.framealpha'] = 1.0

    plt.rcParams['font.size'] = 9              # was 13
    plt.rcParams['xtick.labelsize'] = 8        # was 12
    plt.rcParams['ytick.labelsize'] = 8        # was 12
    plt.rcParams['legend.fontsize'] = 7         # was 11

    
    # Create figure with exact dimensions
    # fig, ax = plt.subplots(figsize=(5.5, 4.2))
    fig, ax = plt.subplots(figsize=(4.0, 3.0))
    
    # Define colors and markers matching the original plots
    style_map = {
        'GAEttack': {'color': '#1f77b4', 'marker': 'o', 'linestyle': '-'},
        'Nettack': {'color': '#ff7f0e', 'marker': 's', 'linestyle': '-'},
        'SGAttack': {'color': '#2ca02c', 'marker': '^', 'linestyle': '-'},
        'FGA': {'color': '#d62728', 'marker': 'D', 'linestyle': '-'},
        'Random': {'color': '#9467bd', 'marker': 'v', 'linestyle': '-'},
        'GOttack': {'color': '#8c564b', 'marker': 'p', 'linestyle': '-'},
    }
    
    # Default style for unknown attacks
    default_colors = ['#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    default_markers = ['p', 'h', '*', 'X', 'P']
    
    # Get data
    budgets = data.get('budgets', [1, 2, 3, 4, 5, 6, 7])
    attacks = data.get('attacks', {})
    
    # Check if show_legend is specified in data, otherwise use parameter
    show_legend_flag = data.get('show_legend', show_legend)
    
    # Define the order of attacks
    attack_order = ['Random', 'FGA', 'Nettack', 'SGAttack', 'GOttack', 'GAEttack']
    
    # Sort attacks according to the defined order
    sorted_attacks = []
    for attack_name in attack_order:
        if attack_name in attacks:
            sorted_attacks.append((attack_name, attacks[attack_name]))
    
    # Add any remaining attacks not in the predefined order
    for attack_name, values in attacks.items():
        if attack_name not in attack_order:
            sorted_attacks.append((attack_name, values))
    
    # Plot each attack
    color_idx = 0
    for attack_name, values in sorted_attacks:
        if attack_name in style_map:
            style = style_map[attack_name]
        else:
            # Use default style for custom attacks
            style = {
                'color': default_colors[color_idx % len(default_colors)],
                'marker': default_markers[color_idx % len(default_markers)],
                'linestyle': '-'
            }
            color_idx += 1
        
        ax.plot(budgets, values, 
                label=attack_name,
                color=style['color'],
                marker=style['marker'],
                linestyle=style['linestyle'],
                linewidth=2.5,
                markersize=9,
                markeredgewidth=1.0,
                markeredgecolor='white',
                markerfacecolor=style['color'],
                alpha=1.0,
                zorder=3)
    
    # Set labels
    # ax.set_xlabel(data.get('xlabel', 'Perturbation Budget'), fontsize=14, fontweight='bold')
    # ax.set_ylabel(data.get('ylabel', 'Misclassification (%)'), fontsize=14, fontweight='bold')

    ax.set_xlabel(data.get('xlabel', 'Perturbation Budget'), fontsize=9, fontweight='bold')  # was 14
    ax.set_ylabel(data.get('ylabel', 'Misclassification (%)'), fontsize=9, fontweight='bold')  # was 14
    
    # Set title if provided
    if 'title' in data:
        # ax.set_title(data['title'], fontsize=15, fontweight='bold', pad=12)
        ax.set_title(data['title'], fontsize=12, fontweight='bold', pad=10)  # was 15, pad=12
    
    # Set axis limits and ticks
    ax.set_xlim(budgets[0] - 0.2, budgets[-1] + 0.2)
    ax.set_ylim(0, 100)
    ax.set_xticks(budgets)
    ax.set_yticks(np.arange(0, 101, 10))
    
    # Make tick labels bold
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    
    # NO GRID for publication-quality plots
    ax.grid(False)
    
    # Legend
    if show_legend_flag:
        legend = ax.legend(loc='best', 
                          frameon=True, 
                          fancybox=False,
                          shadow=False,
                          fontsize=11,
                          edgecolor='black',
                          facecolor='white',
                          framealpha=1.0,
                          ncol=1,
                          prop={'weight': 'bold', 'size': 7}) # was size: 11
        legend.get_frame().set_linewidth(1.2)
    
    # Spine styling - standard axes on all sides
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color('black')
    
    # Tick parameters - professional appearance
    ax.tick_params(axis='both', which='major', labelsize=9, 
                   width=1.5, length=5, direction='in', 
                   top=True, right=True)
    ax.tick_params(axis='both', which='minor', 
                   width=1.2, length=3, direction='in',
                   top=True, right=True)
    
    # Tight layout with minimal padding
    plt.tight_layout(pad=0.3)
    
    # Save with high quality
    plt.savefig(output_filename, 
                format='pdf', 
                dpi=300, 
                bbox_inches='tight',
                pad_inches=0.05)
    print(f"Plot saved as {output_filename}")
    
    # Also save as PNG for preview
    png_filename = output_filename.replace('.pdf', '.png')
    plt.savefig(png_filename, 
                format='png', 
                dpi=300, 
                bbox_inches='tight',
                pad_inches=0.05)
    print(f"Preview saved as {png_filename}")
    
    plt.close()


# Example usage with JSON data
if __name__ == "__main__":
    
    with open('/home/munem/codes/graph-attack/GIN_plot_data_json.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    data_cora = data['cora']
    data_polblogs = data['polblogs']
    data_citeseer = data['citeseer']
    data_blogcatalog = data['blogcatalog']

    # Create plots
    create_attack_plot(data_cora, 'miss_classification_plot_gin_cora.pdf')
    create_attack_plot(data_polblogs, 'miss_classification_plot_gin_polblogs.pdf')
    create_attack_plot(data_citeseer, 'miss_classification_plot_gin_citeseer.pdf')
    create_attack_plot(data_blogcatalog, 'miss_classification_plot_gin_blogcatalog.pdf')  # Uses show_legend from data

Plot saved as miss_classification_plot_gin_cora.pdf
Preview saved as miss_classification_plot_gin_cora.png
Plot saved as miss_classification_plot_gin_polblogs.pdf
Preview saved as miss_classification_plot_gin_polblogs.png
Plot saved as miss_classification_plot_gin_citeseer.pdf
Preview saved as miss_classification_plot_gin_citeseer.png
Plot saved as miss_classification_plot_gin_blogcatalog.pdf
Preview saved as miss_classification_plot_gin_blogcatalog.png
